CSci 167 Project
All code will be here and separated in sections

---



Objectives for thie project: <br>
Compare multiple deep learning architectures
Evaluate how model capacity affects performance by training a baseline CNN, a deeper CNN, and a modern residual network (ResNet-18).<br>

Analyze cross-dataset generalization
Test each model on two datasets of increasing difficulty (CIFAR-10 and CIFAR-100) to understand how architectures scale when classification complexity increases. <br>

Perform structured hyperparameter experimentation
Explore optimizers (SGD vs Adam), learning rates, and regularization settings to determine how sensitive each model is to hyperparameters across datasets. <br>

Identify patterns in training stability and convergence
Observe how learning dynamics (loss/accuracy curves) differ between models and datasets, and which configurations lead to stable or unstable training. <br>

Draw conclusions about model design and training strategy
Determine which combinations of architecture + optimizer + hyperparameters generalize best and why. <br>

In [ ]:
#Colab setup: Load CIFAR-10 From Hugging Face

!pip install -q datasets

import torch
from torch import nn, optim
from torch.utils.data import DataLoader
from torchvision import transforms
from datasets import load_dataset

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


Using device: cuda


In [ ]:
#Loading CIFAR-10 from Hugging Face
hf_ds = load_dataset("cifar10")

#Defining transforms (basic to start)
train_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.4914, 0.4822, 0.4465],
                         std=[0.2023, 0.1994, 0.2010]),
])

test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.4914, 0.4822, 0.4465],
                         std=[0.2023, 0.1994, 0.2010]),
])

#Wrap HF dataset into a PyTorch-compatible object

class HFCIFAR10(torch.utils.data.Dataset):
    def __init__(self, split, transform=None):
        self.ds = hf_ds[split]
        self.transform = transform

    def __len__(self):
        return len(self.ds)

    def __getitem__(self, idx):
        item = self.ds[idx]
        img = item["img"]
        label = item["label"]
        if self.transform:
            img = self.transform(img)
        return img, label

train_dataset = HFCIFAR10("train", transform=train_transform)
test_dataset  = HFCIFAR10("test",  transform=test_transform)

batch_size = 128

train_loader = DataLoader(train_dataset, batch_size=batch_size,
                          shuffle=True, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=batch_size,
                          shuffle=False, num_workers=2, pin_memory=True)

len(train_dataset), len(test_dataset)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/120M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/23.9M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/50000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/10000 [00:00<?, ? examples/s]

(50000, 10000)

Model A: A Small CNN Baseline

In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),  # Sizes of color images: 32x32 -> 16x16

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),  # 16x16 -> 8x8

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),  # 8x8 -> 4x4
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 4 * 4, 256),
            nn.ReLU(),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

model = SimpleCNN(num_classes=10).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.1, momentum=0.9)


Training Loop for Model A

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    running_loss, correct, total = 0.0, 0, 0

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        _, preds = outputs.max(1)
        correct += preds.eq(labels).sum().item()
        total += labels.size(0)

    return running_loss / total, correct / total

@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    running_loss, correct, total = 0.0, 0, 0

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        loss = criterion(outputs, labels)

        running_loss += loss.item() * images.size(0)
        _, preds = outputs.max(1)
        correct += preds.eq(labels).sum().item()
        total += labels.size(0)

    return running_loss / total, correct / total

num_epochs = 10
for epoch in range(1, num_epochs + 1):
    train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion)
    test_loss, test_acc   = evaluate(model, test_loader, criterion)

    print(f"Epoch {epoch:02d}: "
          f"train_loss={train_loss:.4f}, train_acc={train_acc:.4f}, "
          f"test_loss={test_loss:.4f}, test_acc={test_acc:.4f}")


Epoch 01: train_loss=1.5895, train_acc=0.4201, test_loss=1.3982, test_acc=0.5050
Epoch 02: train_loss=1.2818, train_acc=0.5492, test_loss=1.2913, test_acc=0.5508
Epoch 03: train_loss=1.2039, train_acc=0.5845, test_loss=1.2386, test_acc=0.5731
Epoch 04: train_loss=1.1858, train_acc=0.5958, test_loss=1.2084, test_acc=0.5892
Epoch 05: train_loss=1.1708, train_acc=0.6063, test_loss=1.3011, test_acc=0.5686
Epoch 06: train_loss=1.2041, train_acc=0.5974, test_loss=1.2823, test_acc=0.5765
Epoch 07: train_loss=1.1786, train_acc=0.6086, test_loss=1.3158, test_acc=0.5622
Epoch 08: train_loss=1.1860, train_acc=0.6091, test_loss=1.3075, test_acc=0.5732
Epoch 09: train_loss=1.2575, train_acc=0.5892, test_loss=1.4061, test_acc=0.5546
Epoch 10: train_loss=1.2810, train_acc=0.5820, test_loss=1.3668, test_acc=0.5583


Next step is to implement Model B with Deep CNN + Batchnorm + Dropout:
This model tests the effect of using deeper layers, regularization, and larger feature maps

In [ ]:
class DeepCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),  # 32 > 16

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2),  # 16 > 8

            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.Conv2d(256, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.MaxPool2d(2),  # 8 > 4
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.5),
            nn.Linear(256 * 4 * 4, 512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        return self.classifier(self.features(x))


Training loop for Model B

In [ ]:
model_b = DeepCNN(num_classes=10).to(device)
optimizer_b = optim.Adam(model_b.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

num_epochs = 10

print("Training Model B (Deep CNN)...")
for epoch in range(1, num_epochs + 1):
    train_loss, train_acc = train_one_epoch(model_b, train_loader, optimizer_b, criterion)
    test_loss, test_acc   = evaluate(model_b, test_loader, criterion)

    print(f"Epoch {epoch:02d}: "
          f"train_loss={train_loss:.4f}, train_acc={train_acc:.4f}, "
          f"test_loss={test_loss:.4f}, test_acc={test_acc:.4f}")


Training Model B (Deep CNN)...
Epoch 01: train_loss=1.4165, train_acc=0.4829, test_loss=1.0726, test_acc=0.6116
Epoch 02: train_loss=0.9715, train_acc=0.6574, test_loss=0.8062, test_acc=0.7179
Epoch 03: train_loss=0.7921, train_acc=0.7230, test_loss=0.7570, test_acc=0.7427
Epoch 04: train_loss=0.6894, train_acc=0.7623, test_loss=0.6986, test_acc=0.7628
Epoch 05: train_loss=0.6069, train_acc=0.7939, test_loss=0.5952, test_acc=0.7968
Epoch 06: train_loss=0.5346, train_acc=0.8174, test_loss=0.5471, test_acc=0.8148
Epoch 07: train_loss=0.4734, train_acc=0.8382, test_loss=0.6028, test_acc=0.8069
Epoch 08: train_loss=0.4163, train_acc=0.8599, test_loss=0.5389, test_acc=0.8244
Epoch 09: train_loss=0.3753, train_acc=0.8731, test_loss=0.5747, test_acc=0.8227
Epoch 10: train_loss=0.3340, train_acc=0.8858, test_loss=0.5585, test_acc=0.8291


Model C - ResNet-18 for CIFAR-10

In [ ]:
from torchvision.models import resnet18

class ResNet18_CIFAR(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.model = resnet18(weights=None)
        # Adjust first conv layer for 32x32 input
        self.model.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
        self.model.maxpool = nn.Identity()
        self.model.fc = nn.Linear(512, num_classes)

    def forward(self, x):
        return self.model(x)

model_c = ResNet18_CIFAR().to(device)
optimizer_c = optim.SGD(model_c.parameters(), lr=0.1, momentum=0.9, weight_decay=5e-4)
criterion = nn.CrossEntropyLoss()

num_epochs = 10
print("Training Model C (ResNet18)...")
for epoch in range(1, num_epochs + 1):
    train_loss, train_acc = train_one_epoch(model_c, train_loader, optimizer_c, criterion)
    test_loss, test_acc   = evaluate(model_c, test_loader, criterion)

    print(f"Epoch {epoch:02d}: "
          f"train_loss={train_loss:.4f}, train_acc={train_acc:.4f}, "
          f"test_loss={test_loss:.4f}, test_acc={test_acc:.4f}")



Training Model C (ResNet18)...
Epoch 01: train_loss=1.9100, train_acc=0.3228, test_loss=1.5005, test_acc=0.4501
Epoch 02: train_loss=1.3004, train_acc=0.5259, test_loss=1.2342, test_acc=0.5597
Epoch 03: train_loss=0.9740, train_acc=0.6503, test_loss=1.1259, test_acc=0.6061
Epoch 04: train_loss=0.7480, train_acc=0.7367, test_loss=0.7818, test_acc=0.7235
Epoch 05: train_loss=0.6009, train_acc=0.7907, test_loss=0.7511, test_acc=0.7419
Epoch 06: train_loss=0.4912, train_acc=0.8298, test_loss=0.7389, test_acc=0.7579
Epoch 07: train_loss=0.4264, train_acc=0.8531, test_loss=0.6589, test_acc=0.7777
Epoch 08: train_loss=0.3769, train_acc=0.8682, test_loss=0.7002, test_acc=0.7799
Epoch 09: train_loss=0.3371, train_acc=0.8830, test_loss=0.6684, test_acc=0.7804
Epoch 10: train_loss=0.3121, train_acc=0.8906, test_loss=0.6941, test_acc=0.7812


Hyperparameter Experiements:
We will run experiments with SGD and Adam Optimizers, changing the learning rates from 0.1, 0.01, 0.001, changing batch sizes from 64, 128, 256, and adjusting Regularization: Dropout on/off, Weight Decay

In [ ]:
def run_experiment(model_class, config, epochs=10):
    model = model_class().to(device)

    # Optimizer
    if config["optimizer"] == "sgd":
        optimizer = optim.SGD(
            model.parameters(),
            lr=config["lr"],
            momentum=0.9,
            weight_decay=config["weight_decay"],
        )
    else:
        optimizer = optim.Adam(
            model.parameters(),
            lr=config["lr"],
            weight_decay=config["weight_decay"],
        )

    history = {"train_acc": [], "test_acc": [], "train_loss": [], "test_loss": []}

    for epoch in range(epochs):
        train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion)
        test_loss, test_acc = evaluate(model, test_loader, criterion)

        history["train_acc"].append(train_acc)
        history["test_acc"].append(test_acc)
        history["train_loss"].append(train_loss)
        history["test_loss"].append(test_loss)

    return history


In [ ]:
import matplotlib.pyplot as plt

def plot_learning_curve(history, title):
    epochs = range(len(history["train_loss"]))
    plt.figure(figsize=(8,4))
    plt.plot(epochs, history["train_loss"], label="Train Loss")
    plt.plot(epochs, history["test_loss"], label="Test Loss")
    plt.legend()
    plt.title(title)
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.show()


Hyperparameter Framework: Experiment runner

In [ ]:
import pandas as pd

def run_experiment(
    model_class,
    exp_name,
    optimizer_name="sgd",
    lr=0.1,
    weight_decay=5e-4,
    batch_size=128,
    epochs=10,
):
    print(f"\n=== Experiment: {exp_name} ===")
    print(f"Model={model_class.__name__}, opt={optimizer_name}, lr={lr}, "
          f"wd={weight_decay}, batch={batch_size}")

    # Rebuild data loaders for this batch size
    train_loader = DataLoader(
        train_dataset, batch_size=batch_size,
        shuffle=True, num_workers=2, pin_memory=True
    )
    test_loader = DataLoader(
        test_dataset, batch_size=batch_size,
        shuffle=False, num_workers=2, pin_memory=True
    )


    model = model_class(num_classes=10).to(device)
    criterion = nn.CrossEntropyLoss()

    # Optimizer choice
    if optimizer_name == "sgd":
        optimizer = optim.SGD(
            model.parameters(),
            lr=lr,
            momentum=0.9,
            weight_decay=weight_decay,
        )
    elif optimizer_name == "adam":
        optimizer = optim.Adam(
            model.parameters(),
            lr=lr,
            weight_decay=weight_decay,
        )
    else:
        raise ValueError("optimizer_name must be 'sgd' or 'adam'")

    history = {"train_loss": [], "train_acc": [],
               "test_loss": [], "test_acc": []}

    for epoch in range(1, epochs + 1):
        train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion)
        test_loss, test_acc   = evaluate(model, test_loader, criterion)

        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["test_loss"].append(test_loss)
        history["test_acc"].append(test_acc)

        print(f"Epoch {epoch:02d}: "
              f"train_loss={train_loss:.4f}, train_acc={train_acc:.4f}, "
              f"test_loss={test_loss:.4f}, test_acc={test_acc:.4f}")

    # Collect summary metrics for table
    metrics = {
        "exp_name": exp_name,
        "model": model_class.__name__,
        "optimizer": optimizer_name,
        "lr": lr,
        "weight_decay": weight_decay,
        "batch_size": batch_size,
        "final_test_acc": history["test_acc"][-1],
        "best_test_acc": max(history["test_acc"]),
    }
    return history, metrics


Small but meaningful set of experiments:<br> To keep runtime at a reasonable rate, we will start with mini grid and adjust some hyperparameters as mentioned above: <br>Optimmizers: SGD, Adam <br>
Learning rate: 0.1, 0.01, 0.001<br>
Weight Decay: Fixed at 5e-4 (for now)<br>
Batch Size: 128

In [ ]:
deepcnn_results = []
deepcnn_histories = {}

lrs = [0.1, 0.01, 0.001]
opts = ["sgd", "adam"]

for opt in opts:
    for lr in lrs:
        exp_name = f"DeepCNN_{opt}_lr{lr}"
        hist, metrics = run_experiment(
            DeepCNN,
            exp_name,
            optimizer_name=opt,
            lr=lr,
            weight_decay=5e-4,
            batch_size=128,
            epochs=10,
        )
        deepcnn_histories[exp_name] = hist
        deepcnn_results.append(metrics)

# Organize results in a nice viewable table
deepcnn_df = pd.DataFrame(deepcnn_results)
deepcnn_df.sort_values("final_test_acc", ascending=False, inplace=True)
deepcnn_df



=== Experiment: DeepCNN_sgd_lr0.1 ===
Model=DeepCNN, opt=sgd, lr=0.1, wd=0.0005, batch=128
Epoch 01: train_loss=2.0630, train_acc=0.2027, test_loss=1.8011, test_acc=0.3103
Epoch 02: train_loss=1.6388, train_acc=0.3852, test_loss=1.5890, test_acc=0.4526
Epoch 03: train_loss=1.2883, train_acc=0.5401, test_loss=1.1165, test_acc=0.6125
Epoch 04: train_loss=1.0470, train_acc=0.6386, test_loss=0.8533, test_acc=0.7075
Epoch 05: train_loss=0.9193, train_acc=0.6885, test_loss=0.8830, test_acc=0.6943
Epoch 06: train_loss=0.8612, train_acc=0.7165, test_loss=0.9984, test_acc=0.6565
Epoch 07: train_loss=0.7893, train_acc=0.7381, test_loss=0.9438, test_acc=0.6872
Epoch 08: train_loss=0.7338, train_acc=0.7581, test_loss=0.7504, test_acc=0.7467
Epoch 09: train_loss=0.6885, train_acc=0.7755, test_loss=0.7456, test_acc=0.7477
Epoch 10: train_loss=0.6585, train_acc=0.7823, test_loss=0.8472, test_acc=0.7165

=== Experiment: DeepCNN_sgd_lr0.01 ===
Model=DeepCNN, opt=sgd, lr=0.01, wd=0.0005, batch=128
Epoc

,exp_name,model,optimizer,lr,weight_decay,batch_size,final_test_acc,best_test_acc
1,DeepCNN_sgd_lr0.01,DeepCNN,sgd,0.010,0.0005,128,0.8356,0.8356
5,DeepCNN_adam_lr0.001,DeepCNN,adam,0.001,0.0005,128,0.7975,0.8096
2,DeepCNN_sgd_lr0.001,DeepCNN,sgd,0.001,0.0005,128,0.7732,0.7767
0,DeepCNN_sgd_lr0.1,DeepCNN,sgd,0.100,0.0005,128,0.7165,0.7477
4,DeepCNN_adam_lr0.01,DeepCNN,adam,0.010,0.0005,128,0.5728,0.6464
3,DeepCNN_adam_lr0.1,DeepCNN,adam,0.100,0.0005,128,0.1000,0.1000


2nd Data set: CIFAR-100 <br>
We will be performing the same set of tests as CIFAR-10 and compare results in the report/slides


In [ ]:
from datasets import load_dataset

hf_c100 = load_dataset("cifar100")
hf_c100

class HFCIFAR100(torch.utils.data.Dataset):
    def __init__(self, split, transform=None):
        self.ds = hf_c100[split]
        self.transform = transform

    def __len__(self):
        return len(self.ds)

    def __getitem__(self, idx):
        item = self.ds[idx]
        img = item["img"]
        label = item["fine_label"]
        if self.transform:
            img = self.transform(img)
        return img, label


README.md: 0.00B [00:00, ?B/s]

cifar100/train-00000-of-00001.parquet:   0%|          | 0.00/119M [00:00<?, ?B/s]

cifar100/test-00000-of-00001.parquet:   0%|          | 0.00/23.8M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/50000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/10000 [00:00<?, ? examples/s]

In [ ]:
train_c100 = HFCIFAR100("train", transform=train_transform)
test_c100  = HFCIFAR100("test",  transform=test_transform)

train_loader_c100 = DataLoader(train_c100, batch_size=128, shuffle=True, num_workers=2, pin_memory=True)
test_loader_c100  = DataLoader(test_c100, batch_size=128, shuffle=False, num_workers=2, pin_memory=True)

len(train_c100), len(test_c100)


(50000, 10000)

Model A framework and test loops with CIFAR-100 dataset

In [ ]:
modelA_c100 = SimpleCNN(num_classes=100).to(device)
optimizerA_c100 = optim.SGD(modelA_c100.parameters(), lr=0.1, momentum=0.9)
criterion = nn.CrossEntropyLoss()

num_epochs = 10

print("Training SimpleCNN on CIFAR-100...")
for epoch in range(1, num_epochs + 1):
    train_loss, train_acc = train_one_epoch(modelA_c100, train_loader_c100, optimizerA_c100, criterion)
    test_loss, test_acc   = evaluate(modelA_c100, test_loader_c100, criterion)

    print(f"Epoch {epoch:02d}: "
          f"train_loss={train_loss:.4f}, train_acc={train_acc:.4f}, "
          f"test_loss={test_loss:.4f}, test_acc={test_acc:.4f}")


Training SimpleCNN on CIFAR-100...
Epoch 01: train_loss=3.8405, train_acc=0.1133, test_loss=3.4773, test_acc=0.1743
Epoch 02: train_loss=3.3211, train_acc=0.2039, test_loss=3.2559, test_acc=0.2209
Epoch 03: train_loss=3.1621, train_acc=0.2339, test_loss=3.1894, test_acc=0.2309
Epoch 04: train_loss=3.0909, train_acc=0.2504, test_loss=3.2157, test_acc=0.2397
Epoch 05: train_loss=3.0370, train_acc=0.2610, test_loss=3.1806, test_acc=0.2346
Epoch 06: train_loss=2.9969, train_acc=0.2687, test_loss=3.0516, test_acc=0.2684
Epoch 07: train_loss=2.9684, train_acc=0.2791, test_loss=3.2771, test_acc=0.2325
Epoch 08: train_loss=2.9505, train_acc=0.2791, test_loss=3.2840, test_acc=0.2319
Epoch 09: train_loss=2.8961, train_acc=0.2925, test_loss=3.2709, test_acc=0.2376
Epoch 10: train_loss=2.8946, train_acc=0.2948, test_loss=3.3520, test_acc=0.2327


Model B framework and test loops with CIFAR-100

In [ ]:
modelB_c100 = DeepCNN(num_classes=100).to(device)
optimizerB_c100 = optim.SGD(modelB_c100.parameters(), lr=0.1, momentum=0.9, weight_decay=5e-4)
criterion = nn.CrossEntropyLoss()

num_epochs = 10

print("Training DeepCNN on CIFAR-100...")
for epoch in range(1, num_epochs + 1):
    train_loss, train_acc = train_one_epoch(modelB_c100, train_loader_c100, optimizerB_c100, criterion)
    test_loss, test_acc   = evaluate(modelB_c100, test_loader_c100, criterion)

    print(f"Epoch {epoch:02d}: "
          f"train_loss={train_loss:.4f}, train_acc={train_acc:.4f}, "
          f"test_loss={test_loss:.4f}, test_acc={test_acc:.4f}")


Training DeepCNN on CIFAR-100...
Epoch 01: train_loss=4.4259, train_acc=0.0278, test_loss=4.1095, test_acc=0.0564
Epoch 02: train_loss=4.1255, train_acc=0.0544, test_loss=3.8547, test_acc=0.0988
Epoch 03: train_loss=3.8530, train_acc=0.0948, test_loss=3.5412, test_acc=0.1479
Epoch 04: train_loss=3.5671, train_acc=0.1428, test_loss=3.1485, test_acc=0.2223
Epoch 05: train_loss=3.2977, train_acc=0.1852, test_loss=2.9887, test_acc=0.2455
Epoch 06: train_loss=3.0870, train_acc=0.2228, test_loss=2.8469, test_acc=0.2717
Epoch 07: train_loss=2.9158, train_acc=0.2545, test_loss=2.5799, test_acc=0.3335
Epoch 08: train_loss=2.7937, train_acc=0.2777, test_loss=2.5430, test_acc=0.3387
Epoch 09: train_loss=2.6929, train_acc=0.3004, test_loss=2.4633, test_acc=0.3529
Epoch 10: train_loss=2.6144, train_acc=0.3198, test_loss=2.7184, test_acc=0.3242


Model C framework and test loops with CIFAR-100

In [ ]:
modelC_c100 = ResNet18_CIFAR(num_classes=100).to(device)
optimizerC_c100 = optim.SGD(modelC_c100.parameters(), lr=0.1, momentum=0.9, weight_decay=5e-4)
criterion = nn.CrossEntropyLoss()

num_epochs = 10

print("Training ResNet18 on CIFAR-100...")
for epoch in range(1, num_epochs + 1):
    train_loss, train_acc = train_one_epoch(modelC_c100, train_loader_c100, optimizerC_c100, criterion)
    test_loss, test_acc   = evaluate(modelC_c100, test_loader_c100, criterion)

    print(f"Epoch {epoch:02d}: "
          f"train_loss={train_loss:.4f}, train_acc={train_acc:.4f}, "
          f"test_loss={test_loss:.4f}, test_acc={test_acc:.4f}")


Training ResNet18 on CIFAR-100...
Epoch 01: train_loss=3.7739, train_acc=0.1219, test_loss=3.3061, test_acc=0.1950
Epoch 02: train_loss=2.9179, train_acc=0.2633, test_loss=2.7643, test_acc=0.2959
Epoch 03: train_loss=2.2738, train_acc=0.3926, test_loss=2.1849, test_acc=0.4112
Epoch 04: train_loss=1.8453, train_acc=0.4922, test_loss=2.0943, test_acc=0.4400
Epoch 05: train_loss=1.5771, train_acc=0.5525, test_loss=1.9209, test_acc=0.4762
Epoch 06: train_loss=1.3722, train_acc=0.6057, test_loss=1.8825, test_acc=0.5003
Epoch 07: train_loss=1.2034, train_acc=0.6474, test_loss=1.9762, test_acc=0.4803
Epoch 08: train_loss=1.0815, train_acc=0.6786, test_loss=2.1204, test_acc=0.4715
Epoch 09: train_loss=0.9535, train_acc=0.7157, test_loss=1.9715, test_acc=0.4925
Epoch 10: train_loss=0.8468, train_acc=0.7429, test_loss=2.1634, test_acc=0.4783


Hyperparameter Setup for CIFAR-100 dataset<br>
Many of the CIFAR-10 hyperparameters will be mirrored but for CIFAR-100 DeepCNN

In [ ]:
import pandas as pd

def run_experiment_c100(
    model_class,
    exp_name,
    optimizer_name="sgd",
    lr=0.1,
    weight_decay=5e-4,
    batch_size=128,
    epochs=10,
):
    print(f"\n=== CIFAR-100 Experiment: {exp_name} ===")
    print(f"Model={model_class.__name__}, opt={optimizer_name}, lr={lr}, "
          f"wd={weight_decay}, batch={batch_size}")

    # Data loaders for this batch size (CIFAR-100)
    train_loader = DataLoader(
        train_c100, batch_size=batch_size,
        shuffle=True, num_workers=2, pin_memory=True
    )
    test_loader = DataLoader(
        test_c100, batch_size=batch_size,
        shuffle=False, num_workers=2, pin_memory=True
    )

    model = model_class(num_classes=100).to(device)
    criterion = nn.CrossEntropyLoss()

    # Optimizer choice
    if optimizer_name == "sgd":
        optimizer = optim.SGD(
            model.parameters(),
            lr=lr,
            momentum=0.9,
            weight_decay=weight_decay,
        )
    elif optimizer_name == "adam":
        optimizer = optim.Adam(
            model.parameters(),
            lr=lr,
            weight_decay=weight_decay,
        )
    else:
        raise ValueError("optimizer_name must be 'sgd' or 'adam'")

    history = {"train_loss": [], "train_acc": [],
               "test_loss": [], "test_acc": []}

    for epoch in range(1, epochs + 1):
        train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion)
        test_loss, test_acc   = evaluate(model, test_loader, criterion)

        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["test_loss"].append(test_loss)
        history["test_acc"].append(test_acc)

        print(f"Epoch {epoch:02d}: "
              f"train_loss={train_loss:.4f}, train_acc={train_acc:.4f}, "
              f"test_loss={test_loss:.4f}, test_acc={test_acc:.4f}")

    # Summary row for table
    metrics = {
        "exp_name": exp_name,
        "model": model_class.__name__,
        "optimizer": optimizer_name,
        "lr": lr,
        "weight_decay": weight_decay,
        "batch_size": batch_size,
        "final_test_acc": history["test_acc"][-1],
        "best_test_acc": max(history["test_acc"]),
    }
    return history, metrics


To keep runtime reasonable, we will do 4 configurations for DeepCNN <br>
SGD: L.R. = 0.1, 0.01 <br>
Adam: L.R. = 0.0001, 0.0005

In [ ]:
deepcnn_c100_results = []
deepcnn_c100_histories = {}

configs = [
    ("sgd", 0.1),
    ("sgd", 0.01),
    ("adam", 0.001),
    ("adam", 0.0005),
]

for opt, lr in configs:
    exp_name = f"DeepCNN_C100_{opt}_lr{lr}"
    hist, metrics = run_experiment_c100(
        DeepCNN,
        exp_name,
        optimizer_name=opt,
        lr=lr,
        weight_decay=5e-4,
        batch_size=128,
        epochs=10,
    )
    deepcnn_c100_histories[exp_name] = hist
    deepcnn_c100_results.append(metrics)

deepcnn_c100_df = pd.DataFrame(deepcnn_c100_results)
deepcnn_c100_df.sort_values("final_test_acc", ascending=False, inplace=True)
deepcnn_c100_df



=== CIFAR-100 Experiment: DeepCNN_C100_sgd_lr0.1 ===
Model=DeepCNN, opt=sgd, lr=0.1, wd=0.0005, batch=128
Epoch 01: train_loss=4.5669, train_acc=0.0146, test_loss=4.4724, test_acc=0.0248
Epoch 02: train_loss=4.4323, train_acc=0.0229, test_loss=4.2321, test_acc=0.0448
Epoch 03: train_loss=4.2890, train_acc=0.0373, test_loss=4.2210, test_acc=0.0420
Epoch 04: train_loss=4.1546, train_acc=0.0493, test_loss=3.9445, test_acc=0.0816
Epoch 05: train_loss=3.9569, train_acc=0.0740, test_loss=3.6156, test_acc=0.1247
Epoch 06: train_loss=3.7440, train_acc=0.1004, test_loss=3.3801, test_acc=0.1753
Epoch 07: train_loss=3.4789, train_acc=0.1443, test_loss=3.1811, test_acc=0.2038
Epoch 08: train_loss=3.2330, train_acc=0.1845, test_loss=3.0507, test_acc=0.2384
Epoch 09: train_loss=3.0407, train_acc=0.2260, test_loss=2.9666, test_acc=0.2548
Epoch 10: train_loss=2.8807, train_acc=0.2570, test_loss=2.5835, test_acc=0.3192

=== CIFAR-100 Experiment: DeepCNN_C100_sgd_lr0.01 ===
Model=DeepCNN, opt=sgd, lr=0

,exp_name,model,optimizer,lr,weight_decay,batch_size,final_test_acc,best_test_acc
3,DeepCNN_C100_adam_lr0.0005,DeepCNN,adam,0.0005,0.0005,128,0.5356,0.5356
1,DeepCNN_C100_sgd_lr0.01,DeepCNN,sgd,0.0100,0.0005,128,0.5010,0.5090
2,DeepCNN_C100_adam_lr0.001,DeepCNN,adam,0.0010,0.0005,128,0.4837,0.4837
0,DeepCNN_C100_sgd_lr0.1,DeepCNN,sgd,0.1000,0.0005,128,0.3192,0.3192
